# Predicción de Jugadas en Póker Texas Hold'em con IA
## Comparativa: Minimax · Red Bayesiana · Cadenas de Markov · TD Learning

---

## Tabla de Contenidos

- [Sección 0: Instalación y Configuración](#seccion-0)
- [Sección 1: Fundamentos — Representación del Juego](#seccion-1)
- [Sección 2: Modelo 1 — Minimax con Poda Alpha-Beta](#seccion-2)
- [Sección 3: Modelo 2 — Red Bayesiana](#seccion-3)
- [Sección 4: Modelo 3 — Cadenas de Markov](#seccion-4)
- [Sección 5: Modelo 4 — TD Learning (Q-Learning)](#seccion-5)
- [Sección 6: Modelo Combinado (Ensemble)](#seccion-6)
- [Sección 7: Comparación y Evaluación](#seccion-7)

<a id='seccion-0'></a>
## Sección 0: Instalación y Configuración

Instalamos todas las dependencias necesarias y configuramos el entorno de ejecución con una semilla aleatoria fija para garantizar reproducibilidad.


In [1]:
# Importaciones globales
import random
import time
import warnings
import itertools
from copy import deepcopy
from typing import List, Tuple, Dict, Optional, Any
from enum import Enum, auto
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Evaluador de manos de póker
from treys import Card, Evaluator, Deck

# Red Bayesiana
from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.estimators import MaximumLikelihoodEstimator

warnings.filterwarnings('ignore')

# Semilla global
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Entorno configurado correctamente.")
print(f"   Semilla aleatoria global: {RANDOM_STATE}")


c:\Users\gabri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gabri\AppData\Local\Programs\Python\Python312\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Entorno configurado correctamente.
   Semilla aleatoria global: 42


<a id='seccion-1'></a>
## Sección 1: Fundamentos - Representación del Juego

Antes de implementar cualquier modelo de IA, necesitamos una representación uniforme del juego. Todos los agentes operarán sobre la misma clase `GameState`, lo que garantiza una comparación justa.

### Componentes del entorno:
1. **Mazo y cartas**: representadas en formato compatible con `treys`
2. **Estado del juego**: encapsula toda la información necesaria para tomar una decisión
3. **Espacio de acciones**: FOLD, CHECK, CALL, RAISE (3 tamaños), ALL_IN
4. **Evaluación de manos**: usando `treys` + estimación de equity por Monte Carlo


In [2]:
# 1.1  Enumeraciones y constantes del juego

class Phase(Enum): # Fases del juego Texas Hold'em
    PREFLOP  = 0
    FLOP     = 1
    TURN     = 2
    RIVER    = 3
    SHOWDOWN = 4


class Action(Enum): # Acciones disponibles para un jugador
    FOLD        = 0
    CHECK       = 1
    CALL        = 2
    RAISE_HALF  = 3   # 0.5× pot
    RAISE_POT   = 4   # 1×  pot
    RAISE_2POT  = 5   # 2×  pot
    ALL_IN      = 6


# Mapeo legible de acción → descripción
ACTION_LABELS = {
    Action.FOLD:       "Fold",
    Action.CHECK:      "Check",
    Action.CALL:       "Call",
    Action.RAISE_HALF: "Raise ½ pot",
    Action.RAISE_POT:  "Raise 1x pot",
    Action.RAISE_2POT: "Raise 2x pot",
    Action.ALL_IN:     "All-in",
}

# Score numérico para el ensemble (agresividad creciente)
ACTION_SCORES = {
    Action.FOLD:       0,
    Action.CHECK:      1,
    Action.CALL:       2,
    Action.RAISE_HALF: 3,
    Action.RAISE_POT:  4,
    Action.RAISE_2POT: 5,
    Action.ALL_IN:     6,
}

STARTING_STACK = 1000   # fichas iniciales por jugador
BIG_BLIND      = 20     # ciega grande

print("Enumeraciones y constantes definidas.")


Enumeraciones y constantes definidas.


In [3]:
# 1.2  Clase GameState

class GameState:
    """
    Encapsula el estado completo de una mano de Texas Hold'em.

    - Attributes
    hole_cards : List[int]
        Las dos cartas privadas del jugador (formato treys).
    community_cards : List[int]
        Cartas comunitarias visibles (0-5 cartas).
    pot : float
        Fichas acumuladas en el bote.
    stacks : List[float]
        Stack de cada jugador [jugador_0, jugador_1].
    current_bet : float
        Apuesta actual que hay que igualar.
    phase : Phase
        Fase del juego (PREFLOP, FLOP, TURN, RIVER, SHOWDOWN).
    acting_player : int
        Índice del jugador que debe actuar (0 o 1).
    action_history : List[Tuple]
        Historial de acciones (fase, jugador, acción).
    position : int
        Posición del jugador principal (0=EARLY/SB, 1=LATE/BB).
    """

    def __init__(
        self,
        hole_cards: List[int],
        community_cards: Optional[List[int]] = None,
        pot: float = 0.0,
        stacks: Optional[List[float]] = None,
        current_bet: float = 0.0,
        phase: Phase = Phase.PREFLOP,
        acting_player: int = 0,
        position: int = 0,
    ):
        self.hole_cards       = hole_cards
        self.community_cards  = community_cards if community_cards is not None else []
        self.pot              = pot
        self.stacks           = stacks if stacks is not None else [STARTING_STACK, STARTING_STACK]
        self.current_bet      = current_bet
        self.phase            = phase
        self.acting_player    = acting_player
        self.action_history   = []
        self.position         = position

    # Helpers

    @property
    def is_terminal(self) -> bool:
        """True si la mano ha terminado."""
        return self.phase == Phase.SHOWDOWN or any(s <= 0 for s in self.stacks)

    @property
    def pot_odds(self) -> float:
        """Relación call_amount / (pot + call_amount). 0 si no hay apuesta."""
        if self.current_bet <= 0:
            return 0.0
        call_cost = self.current_bet
        return call_cost / (self.pot + call_cost)

    @property
    def stack_ratio(self) -> float:
        """Stack del jugador activo sobre el stack total."""
        total = sum(self.stacks)
        return self.stacks[self.acting_player] / total if total > 0 else 0.5

    def __repr__(self) -> str:
        cards_str = Card.ints_to_pretty_str(self.hole_cards) if self.hole_cards else "[]"
        comm_str  = Card.ints_to_pretty_str(self.community_cards) if self.community_cards else "[]"
        return (
            f"GameState(phase={self.phase.name}, "
            f"hole={cards_str}, community={comm_str}, "
            f"pot={self.pot:.0f}, bet={self.current_bet:.0f}, "
            f"stacks={[int(s) for s in self.stacks]})"
        )


print("GameState definido.")


GameState definido.


In [4]:
# 1.3  Acciones disponibles

def get_legal_actions(state: GameState) -> List[Action]:
    """
    Retorna la lista de acciones legales en el estado actual.

    Reglas:
    - FOLD siempre es legal.
    - CHECK es legal si no hay apuesta pendiente.
    - CALL es legal si hay apuesta pendiente y el jugador tiene fichas.
    - RAISE es legal si el jugador tiene suficientes fichas.
    - ALL_IN siempre es legal si el jugador tiene fichas.

    Parameters, state : GameState

    Returns, List[Action]
    """
    stack   = state.stacks[state.acting_player]
    bet     = state.current_bet
    pot     = state.pot
    actions = [Action.FOLD]

    if bet <= 0:
        actions.append(Action.CHECK)
    else:
        if stack >= bet:
            actions.append(Action.CALL)

    # Raises
    for action, multiplier in [
        (Action.RAISE_HALF, 0.5),
        (Action.RAISE_POT,  1.0),
        (Action.RAISE_2POT, 2.0),
    ]:
        raise_amount = max(multiplier * pot, BIG_BLIND)
        if stack >= bet + raise_amount:
            actions.append(action)

    # ALL_IN
    if stack > 0:
        actions.append(Action.ALL_IN)

    return actions


def action_to_bet_size(action: Action, state: GameState) -> float: # Convierte una acción a su tamaño de apuesta en fichas
    pot = state.pot
    bet = state.current_bet
    if action == Action.FOLD:        return 0.0
    if action == Action.CHECK:       return 0.0
    if action == Action.CALL:        return bet
    if action == Action.RAISE_HALF:  return bet + max(0.5 * pot, BIG_BLIND)
    if action == Action.RAISE_POT:   return bet + max(1.0 * pot, BIG_BLIND)
    if action == Action.RAISE_2POT:  return bet + max(2.0 * pot, BIG_BLIND)
    if action == Action.ALL_IN:      return state.stacks[state.acting_player]
    return 0.0


print("Espacio de acciones definido.")
print("  Ejemplo de acciones legales en estado vacío:")
demo_state = GameState(hole_cards=[], stacks=[500, 500], current_bet=0, pot=100)
print(" ", [a.name for a in get_legal_actions(demo_state)])


Espacio de acciones definido.
  Ejemplo de acciones legales en estado vacío:
  ['FOLD', 'CHECK', 'RAISE_HALF', 'RAISE_POT', 'RAISE_2POT', 'ALL_IN']


In [5]:
# 1.4  Evaluación de manos y cálculo de equity

EVALUATOR = Evaluator()

# Mazo estándar en formato treys
ALL_CARDS = [Card.new(r + s) for r in "23456789TJQKA" for s in "shdc"]


def normalize_hand_strength(score: int) -> float:
    """
    Normaliza el score de treys a [0, 1].

    treys usa 1 = mejor mano (Royal Flush), 7462 = peor (7-high).
    Invertimos y normalizamos para que 1.0 = la mano más fuerte.

    Parameters, score : int  (1 ≤ score ≤ 7462)

    Returns, float  strength ∈ [0, 1]
    """
    return 1.0 - (score - 1) / 7461.0


def evaluate_hand(hole_cards: List[int], community_cards: List[int]) -> float:
    """
    Evalúa la fuerza normalizada de una mano (0→peor, 1→mejor).
    Requiere al menos 3 cartas comunitarias (flop).
    """
    if len(community_cards) < 3:
        return 0.5   # Sin información suficiente → fuerza neutra
    score = EVALUATOR.evaluate(community_cards, hole_cards)
    return normalize_hand_strength(score)


def simulate_equity(
    hole_cards: List[int],
    community_cards: List[int],
    n_simulations: int = 1000,
) -> float:
    """
    Estima la equity del jugador mediante simulación Monte Carlo.

    Para cada simulación:
    1. Completa el tablero con cartas aleatorias del mazo restante.
    2. Sortea 2 cartas aleatorias para el oponente.
    3. Compara manos.

    Parameters
    hole_cards       : cartas del jugador (formato treys)
    community_cards  : cartas comunitarias ya visibles
    n_simulations    : número de simulaciones

    Returns, float  equity ∈ [0, 1]  (probabilidad estimada de ganar)
    """
    if len(hole_cards) < 2:
        return 0.5

    used    = set(hole_cards + community_cards)
    deck    = [c for c in ALL_CARDS if c not in used]
    wins    = 0
    ties    = 0
    n_board = 5 - len(community_cards)

    for _ in range(n_simulations):
        sample = random.sample(deck, n_board + 2)
        board  = community_cards + sample[:n_board]
        opp    = sample[n_board:]

        try:
            my_score  = EVALUATOR.evaluate(board, hole_cards)
            opp_score = EVALUATOR.evaluate(board, opp)
        except Exception:
            continue

        if my_score < opp_score:        # menor score = mejor mano en treys
            wins += 1
        elif my_score == opp_score:
            ties += 1

    return (wins + 0.5 * ties) / n_simulations


print("Evaluación de manos definida.")

# Prueba rápida
test_hole = [Card.new('As'), Card.new('Kh')]
test_comm = [Card.new('Ac'), Card.new('Ad'), Card.new('2h')]
eq = simulate_equity(test_hole, test_comm, n_simulations=500)
hs = evaluate_hand(test_hole, test_comm)
print(f"  Mano: A♠ K♥ | Tablero: A♣ A♦ 2♥")
print(f"  Fuerza normalizada : {hs:.3f}")
print(f"  Equity estimada    : {eq:.3f}")


Evaluación de manos definida.
  Mano: A♠ K♥ | Tablero: A♣ A♦ 2♥
  Fuerza normalizada : 0.783
  Equity estimada    : 0.969


In [6]:
# 1.5  Simulador de partida

class BaseAgent: # Clase base para todos los agentes
    name: str = "BaseAgent"

    def choose_action(self, state: GameState) -> Action:
        raise NotImplementedError


class RandomAgent(BaseAgent): # Agente que elige acciones uniformemente al azar
    name = "Random"

    def choose_action(self, state: GameState) -> Action:
        return random.choice(get_legal_actions(state))


class CallAgent(BaseAgent): # Agente pasivo: siempre hace call/check; fold solo si es la única opción.
    name = "Call"

    def choose_action(self, state: GameState) -> Action:
        legal = get_legal_actions(state)
        for a in [Action.CALL, Action.CHECK]:
            if a in legal:
                return a
        return legal[0]


def deal_hand(rng: random.Random) -> Tuple[List[int], List[int], List[int]]:
    """
    Reparte una mano completa: 2 hole cards por jugador + 5 comunitarias.
    Returns (hole_p0, hole_p1, community)
    """
    deck = list(ALL_CARDS)
    rng.shuffle(deck)
    return deck[0:2], deck[2:4], deck[4:9]


def play_hand(
    agent0: BaseAgent,
    agent1: BaseAgent,
    rng: Optional[random.Random] = None,
) -> Tuple[int, float]:
    """
    Simula una mano completa entre dos agentes.

    Returns
    winner : int   (0 o 1; -1 si empate)
    profit : float fichas ganadas por el agente 0 (negativo = pierde)
    """
    if rng is None:
        rng = random.Random()

    hole0, hole1, community = deal_hand(rng)
    stacks = [float(STARTING_STACK), float(STARTING_STACK)]

    # Ciegas
    stacks[0] -= BIG_BLIND / 2    # small blind
    stacks[1] -= BIG_BLIND        # big blind
    pot = BIG_BLIND * 1.5
    current_bet = float(BIG_BLIND)

    agents        = [agent0, agent1]
    hole_cards    = [hole0, hole1]
    phases        = [Phase.PREFLOP, Phase.FLOP, Phase.TURN, Phase.RIVER]
    comm_revealed = [0, 3, 4, 5]
    folded        = [False, False]
    winner        = -1
    profit        = 0.0

    for phase_idx, (phase, n_comm) in enumerate(zip(phases, comm_revealed)):
        if any(folded) or any(s <= 0 for s in stacks):
            break

        comm_now = community[:n_comm]
        if phase_idx > 0:
            current_bet = 0.0    # nueva calle

        # Hasta dos acciones por calle (simplificado)
        for turn in range(2):
            actor = turn % 2
            if folded[actor] or stacks[actor] <= 0:
                continue

            state = GameState(
                hole_cards    = hole_cards[actor],
                community_cards = comm_now,
                pot           = pot,
                stacks        = list(stacks),
                current_bet   = current_bet,
                phase         = phase,
                acting_player = actor,
                position      = actor,
            )
            action = agents[actor].choose_action(state)
            bet_size = action_to_bet_size(action, state)

            if action == Action.FOLD:
                folded[actor] = True
                winner = 1 - actor
                profit = (STARTING_STACK - stacks[0]) * (-1 if actor == 0 else 1)
                break
            elif action in (Action.CHECK,):
                pass
            elif action == Action.CALL:
                paid = min(current_bet, stacks[actor])
                stacks[actor] -= paid
                pot += paid
            elif action in (Action.RAISE_HALF, Action.RAISE_POT, Action.RAISE_2POT, Action.ALL_IN):
                paid = min(bet_size, stacks[actor])
                stacks[actor] -= paid
                pot += paid
                current_bet = paid
        else:
            continue
        break

    # Showdown si nadie fold
    if not any(folded):
        score0 = EVALUATOR.evaluate(community[:5], hole0) if len(community) >= 5 else 7462
        score1 = EVALUATOR.evaluate(community[:5], hole1) if len(community) >= 5 else 7462
        if score0 < score1:
            winner = 0
        elif score1 < score0:
            winner = 1
        else:
            winner = -1

        if winner == 0:
            profit = pot - (STARTING_STACK - stacks[0])
        elif winner == 1:
            profit = -(STARTING_STACK - stacks[0])
        else:
            profit = 0.0

    return winner, profit


def play_tournament(
    agents: List[BaseAgent],
    n_hands: int = 500,
    seed: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Torneo round-robin: cada agente juega n_hands manos contra cada otro.

    Returns
    pd.DataFrame con columnas [Agent, Opponent, Wins, Losses, Ties, Profit, WinRate]
    """
    rng     = random.Random(seed)
    records = []

    pairs = list(itertools.combinations(range(len(agents)), 2))
    for i, j in tqdm(pairs, desc="Torneo round-robin"):
        a0, a1 = agents[i], agents[j]
        wins0, wins1, ties0 = 0, 0, 0
        profit0 = 0.0

        for _ in range(n_hands):
            w, p = play_hand(a0, a1, rng)
            profit0 += p
            if w == 0:   wins0 += 1
            elif w == 1: wins1 += 1
            else:        ties0 += 1

        records.append({
            "Agent":    a0.name, "Opponent": a1.name,
            "Wins":     wins0,   "Losses":   wins1,   "Ties": ties0,
            "Profit":   profit0, "WinRate":  wins0 / n_hands,
        })
        records.append({
            "Agent":    a1.name, "Opponent": a0.name,
            "Wins":     wins1,   "Losses":   wins0,   "Ties": ties0,
            "Profit":   -profit0, "WinRate": wins1 / n_hands,
        })

    return pd.DataFrame(records)


print("Simulador de partida definido.")
print("  Prueba: Random vs Call (10 manos):")
rng_test = random.Random(42)
for _ in range(3):
    w, p = play_hand(RandomAgent(), CallAgent(), rng_test)
    print(f"    Ganador: {w} | Profit p0: {p:.0f}")


Simulador de partida definido.
  Prueba: Random vs Call (10 manos):
    Ganador: 1 | Profit p0: -10
    Ganador: 0 | Profit p0: 1000
    Ganador: 1 | Profit p0: -10
